# Protenix — Pilot10 corrected **full-batch** Colab (RNA only, no MSA or templates)

**Use Runtime → Change runtime type → T4 GPU (or L4/A100), then Runtime → Run all.** This notebook processes **all ten frozen RNA targets in one execution** and saves completed valid structures to Google Drive after each target. It is restart-safe: rerunning *Run all* skips targets with verified completion markers, and retries unfinished ones. **No manual cell re-runs or configuration toggles.**

**Why this v2 notebook exists:** the original v1 generated files containing NaN coordinates (0 finite atoms). A controlled `9g4p_B` test succeeded with 705/705 finite atoms when setting `LAYERNORM_TYPE=torch` and conservative FP32/Torch-kernel settings. This corrected pipeline requires **100% finite atom coordinates**, an expected RNA residue count, and finite primary confidence scores. It will stop on the first failed target instead of falsely declaring success.

Scientific condition: same frozen 10 FASTAs as the prior 6-model panel, `protenix_base_default_v1.0.0`, seed 101, one sampled structure, RNA-only; no MSA, RNA-MSA, or native templates. This is an **exploratory single-sequence** condition, not a full-feature Protenix evaluation. Scores are not calculated here. Existing invalid v1 outputs are **not** reused.

## 1 — Mount Drive, fixed experiment configuration (do not change)

In [ ]:
from google.colab import drive
from pathlib import Path
import os, sys, json, time, math, csv, shutil, hashlib, subprocess, datetime

drive.mount('/content/drive')
PROJECT_ROOT = Path('/content/drive/MyDrive/RNA3D_Benchmark')
FROZEN_INPUTS = PROJECT_ROOT / 'NuFold/runs/pilot10_single_20261018T203832Z/inputs'
RUN_ID = 'protenix_pilot10_rna_only_nomsa_notemplate_v2_layernormtorch'
RUN_ROOT = PROJECT_ROOT / 'Protenix/runs' / RUN_ID
UPSTREAM_COMMIT = '85767b811c40ed46e73a9b39519cf6bfca8701ba'
MODEL_NAME = 'protenix_base_default_v1.0.0'
TARGETS = ['8ubd_H','9fia_bJ','9cfn_B','9m31_B','9g4p_B','9lq7_K','8z8u_B','9g6k_lB','9cf2_W','8wf8_B']
EXPECTED_LENGTHS = [36,71,59,51,33,49,41,45,61,66]
SEED, N_SAMPLES, N_CYCLE, N_STEP, DTYPE = 101, 1, 10, 200, 'fp32'
assert len(TARGETS) == len(EXPECTED_LENGTHS) == len(set(TARGETS)) == 10
assert RUN_ROOT != FROZEN_INPUTS and RUN_ID.endswith('v2_layernormtorch')
assert FROZEN_INPUTS.is_dir(), f'Frozen input folder missing: {FROZEN_INPUTS}'
RUN_ROOT.mkdir(parents=True, exist_ok=True)
print('Full-batch corrected Protenix v2: 10 targets / 1 sample per target')
print('Run folder:', RUN_ROOT)
print('Previous v1 predictions are not used by this notebook.')

## 2 — Validate all frozen FASTAs and generate 10 RNA-only JSON requests

In [ ]:
def sha256(path):
    h=hashlib.sha256()
    with Path(path).open('rb') as f:
        for block in iter(lambda:f.read(2**20),b''):h.update(block)
    return h.hexdigest()

def load_fasta(path):
    lines=[line.strip() for line in Path(path).read_text().splitlines() if line.strip()]
    assert lines and lines[0].startswith('>'),f'Bad FASTA: {path}'
    seq=''.join(lines[1:]).upper()
    assert seq and set(seq)<=set('AUGC'),(path,set(seq))
    return seq

INPUT_DIR=Path('/content/protenix_pilot10_v2_inputs')
INPUT_DIR.mkdir(parents=True,exist_ok=True)
sequences={}
for target, length in zip(TARGETS,EXPECTED_LENGTHS):
    fasta=FROZEN_INPUTS/target/(target+'.fasta')
    assert fasta.is_file(),f'Missing frozen FASTA: {fasta}'
    seq=load_fasta(fasta)
    assert len(seq)==length,(target,len(seq),length)
    request=[{'name':target,'sequences':[{'rnaSequence':{'sequence':seq,'count':1}}]}]
    req=INPUT_DIR/f'{target}.json'
    req.write_text(json.dumps(request,indent=2)+'\n')
    sequences[target]={'length':length,'fasta':fasta,'fasta_sha256':sha256(fasta),'json':req}
    print(target,length,'nt',sha256(fasta)[:12])
assert len(sequences)==10
print('FROZEN INPUT CHECK: 10/10')

## 3 — Install frozen Protenix and validate GPU

In [ ]:
import torch
assert torch.cuda.is_available(), 'Set Runtime > Change runtime type > GPU before Run all.'
print('GPU:',torch.cuda.get_device_name(0))
print('GPU VRAM GiB:',round(torch.cuda.get_device_properties(0).total_memory/2**30,2))
# Installation and pinning are the same as the original v1 notebook.
SETUP_LOG=RUN_ROOT/'setup_install.log'
with SETUP_LOG.open('w') as out:
    p=subprocess.run([sys.executable,'-m','pip','install','-q','gemmi',
       'git+https://github.com/bytedance/Protenix.git@'+UPSTREAM_COMMIT],
       stdout=out,stderr=subprocess.STDOUT)
if p.returncode:
    print(SETUP_LOG.read_text()[-7000:])
    raise RuntimeError(f'Protenix installation failed with code {p.returncode}. See {SETUP_LOG}')
import gemmi, importlib.metadata
CLI=shutil.which('protenix')
assert CLI, 'Protenix CLI missing after installation'
print('Installed Protenix version:',importlib.metadata.version('protenix'))
print('Gemmi version:',gemmi.__version__)
print('CLI:',CLI)

## 4 — One-click full-batch inference, strict geometric validation and resume-safe Drive saves

In [ ]:
def strict_cif_stats(path, expected_length):
    structure=gemmi.read_structure(str(path))
    assert len(structure)>=1, 'No structure model found in CIF'
    residues=[res for chain in structure[0] for res in chain if res.entity_type==gemmi.EntityType.Polymer]
    assert len(residues)==expected_length, f'RNA polymer residues {len(residues)} != {expected_length}'
    atoms=[atom for res in residues for atom in res]
    assert atoms, 'No atoms in CIF'
    finite=sum(all(math.isfinite(v) for v in (a.pos.x,a.pos.y,a.pos.z)) for a in atoms)
    assert finite==len(atoms),f'NON-FINITE COORDINATES {finite}/{len(atoms)} atoms. Refusing to save invalid output.'
    return len(residues),len(atoms),finite

def strict_confidence(path):
    confidence=json.loads(Path(path).read_text())
    for key in ('plddt','ptm','ranking_score'):
        val=confidence.get(key)
        assert isinstance(val,(int,float)) and math.isfinite(val),f'Confidence {key} is invalid: {val}'
    return {k:float(confidence[k]) for k in ('plddt','ptm','ranking_score')}

def complete(target):
    folder=RUN_ROOT/target
    marker=folder/'complete.json'
    if not marker.is_file(): return False
    try:
        m=json.loads(marker.read_text())
        assert m['status']=='PASS' and m['target_id']==target
        assert m['upstream_commit']==UPSTREAM_COMMIT and m['model_name']==MODEL_NAME
        assert m['layernorm_type']=='torch' and m['dtype']==DTYPE
        assert m['seed']==SEED and m['n_samples']==N_SAMPLES
        assert m['fasta_sha256']==sequences[target]['fasta_sha256']
        for a in m['artifacts']:
            p=folder/a['filename']
            assert p.is_file() and sha256(p)==a['sha256']
        n,atoms,finite=strict_cif_stats(folder/'prediction.cif',sequences[target]['length'])
        strict_confidence(folder/'summary_confidence.json')
        assert m['cif_residues']==n and m['atoms_total']==atoms and m['atoms_finite']==finite
        return True
    except (AssertionError,ValueError,KeyError,TypeError,OSError,RuntimeError) as exc:
        print('Ignoring invalid/stale completion marker for',target,':',exc)
        return False

def current_status(write=True):
    rows=[]
    for t in TARGETS:
        good=complete(t)
        meta=json.loads((RUN_ROOT/t/'complete.json').read_text()) if good else {}
        rows.append({'target':t,'status':'PASS' if good else 'PENDING',
           'length':sequences[t]['length'],'runtime_seconds':meta.get('runtime_seconds',''),
           'cif_residues':meta.get('cif_residues',''),
           'finite_atoms':meta.get('atoms_finite',''),'total_atoms':meta.get('atoms_total','')})
    if write:
        with (RUN_ROOT/'run_status.csv').open('w',newline='') as fh:
            w=csv.DictWriter(fh,fieldnames=list(rows[0]));w.writeheader();w.writerows(rows)
    return rows

assert len([t for t in TARGETS if not complete(t)])<=10
print('Already verified:',sum(complete(t) for t in TARGETS),'/10')

for number,target in enumerate(TARGETS,1):
    if complete(target):
        print(f'[{number}/10] SKIP {target}: already valid and verified on Drive')
        continue
    target_dir=RUN_ROOT/target
    if target_dir.exists() and any(target_dir.iterdir()):
        # Fail closed rather than overwrite earlier artifacts silently.
        raise RuntimeError(f'Partial/invalid v2 target folder requires review before reuse: {target_dir}')
    local_out=Path('/content/protenix_pilot10_v2_work')/target
    if local_out.exists():shutil.rmtree(local_out) # local Colab scratch only
    local_out.mkdir(parents=True,exist_ok=True)
    print(f'[{number}/10] START {target} ({sequences[target]["length"]} nt); LayerNorm=torch',flush=True)
    cmd=[CLI,'pred','-i',str(sequences[target]['json']),'-o',str(local_out),
         '-n',MODEL_NAME,'-s',str(SEED),'-c',str(N_CYCLE),'-p',str(N_STEP),
         '-e',str(N_SAMPLES),'-d',DTYPE,'--use_default_params','false',
         '--use_msa','false','--use_rna_msa','false','--use_template','false',
         '--trimul_kernel','torch','--triatt_kernel','torch',
         '--enable_cache','false','--enable_fusion','false','--enable_tf32','false']
    env=os.environ.copy()
    env['LAYERNORM_TYPE']='torch'  # CRITICAL tested NaN remediation: subprocess environment
    started=time.monotonic()
    local_log=local_out/'inference.full.log'
    with local_log.open('w') as log:
        p=subprocess.run(cmd,env=env,stdout=log,stderr=subprocess.STDOUT)
    elapsed=round(time.monotonic()-started,3)
    if p.returncode:
        fail=RUN_ROOT/'failures';fail.mkdir(parents=True,exist_ok=True)
        (fail/(target+'_error.txt')).write_text(local_log.read_text(errors='replace')[-40000:])
        raise RuntimeError(f'{target} inference exited {p.returncode}; last log 40000 characters stored in Drive failures/')
    cifs=list(local_out.rglob('*_sample_0.cif'))
    confs=list(local_out.rglob('*_summary_confidence_sample_0.json'))
    assert len(cifs)==len(confs)==1,(target,len(cifs),len(confs))
    try:
        n,atoms,finite=strict_cif_stats(cifs[0],sequences[target]['length'])
        scores=strict_confidence(confs[0])
    except Exception as exc:
        fail=RUN_ROOT/'failures';fail.mkdir(exist_ok=True,parents=True)
        (fail/(target+'_invalid_geometry.txt')).write_text(str(exc)+'\n'+local_log.read_text(errors='replace')[-30000:])
        raise
    target_dir.mkdir(parents=True,exist_ok=False)
    # Preserve finite outputs and concise logs; save complete.json strictly last.
    tail_log=local_out/'inference.log'
    tail_log.write_text(local_log.read_text(errors='replace')[-60000:])
    files={'prediction.cif':cifs[0],'summary_confidence.json':confs[0],
           'inference.log':tail_log,'input.json':sequences[target]['json'],
           'input.fasta':sequences[target]['fasta']}
    artifacts=[]
    for name,source in files.items():
        dest=target_dir/name
        shutil.copy2(source,dest)
        assert sha256(dest)==sha256(source)
        artifacts.append({'filename':name,'sha256':sha256(dest),'bytes':dest.stat().st_size})
    meta={'status':'PASS','model':'Protenix','model_name':MODEL_NAME,
          'upstream_commit':UPSTREAM_COMMIT,'target_id':target,'sequence_length':sequences[target]['length'],
          'fasta_sha256':sequences[target]['fasta_sha256'],
          'condition':'RNA-only, single-sequence, no MSA, no RNA MSA, no templates',
          'native_used_for_inference':False,'seed':SEED,'n_samples':N_SAMPLES,
          'cycle':N_CYCLE,'step':N_STEP,'dtype':DTYPE,'layernorm_type':'torch',
          'trimul_kernel':'torch','triatt_kernel':'torch','cache':False,'fusion':False,'tf32':False,
          'gpu':torch.cuda.get_device_name(0),'runtime_seconds':elapsed,
          'cif_residues':n,'atoms_total':atoms,'atoms_finite':finite,
          'confidence_summary':scores,'artifacts':artifacts,
          'saved_utc':datetime.datetime.now(datetime.timezone.utc).isoformat()}
    (target_dir/'complete.json').write_text(json.dumps(meta,indent=2)+'\n')
    assert complete(target),f'Post-save validation failed for {target}'
    current_status()
    print(f'[{number}/10] PASS {target}: {finite}/{atoms} finite atoms; {elapsed}s; DRIVE SAVED',flush=True)

rows=current_status()
count=sum(x['status']=='PASS' for x in rows)
print('FINAL DRIVE VERIFICATION:',count,'/10')
assert count==10,'Not all 10 targets have valid, verified 3D structures; do not evaluate yet'
(RUN_ROOT/'FINALIZED.json').write_text(json.dumps({
    'status':'complete','target_count':10,'model_name':MODEL_NAME,'layernorm_type':'torch',
    'source_run_id':RUN_ID,'targets':TARGETS,'verified_utc':datetime.datetime.now(datetime.timezone.utc).isoformat(),
    'validation':'all atoms finite, lengths verified, essential confidence finite, SHA256 validated'
},indent=2)+'\n')
print('ALL 10 VALID PROTENIX PREDICTIONS VERIFIED ON DRIVE. Ready for native-mapped US-align evaluation.')

## After Run All

The final line must say **`ALL 10 VALID PROTENIX PREDICTIONS VERIFIED ON DRIVE`**; merely seeing `10/10` from the old v1 notebook is **not** sufficient. You can rerun the entire notebook in another GPU session if Colab disconnects: successfully saved target folders are verified and skipped. Any invalid or partial new-run target folder stops the notebook for safe review, without silently deleting it. US-align is a separate evaluation step using the same frozen experimental native mappings as the six previous models.